In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support


# Settiing up main functions

In [ ]:

def parse_skill_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return []

    seen = set()
    skills = []
    for raw in text.split("|"):
        skill = raw.strip()
        if skill and skill not in seen:
            seen.add(skill)
            skills.append(skill)
    return skills


parse_skill_list(value)
Takes a string like:

"Excel | Auditing | Tax"
and converts it into:

["Excel", "Auditing", "Tax"]

In [ ]:
def load_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    if not records:
        raise ValueError(f"No records found in {path}")
    return records

load_jsonl(path)
Reads a .jsonl file, where each line is one JSON object, and returns a list of records.

In [ ]:
def build_indicator_matrix(skill_lists, skill_names):
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)
    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            indicator[row_idx, skill_to_idx[skill]] = 1
    return indicator


build_indicator_matrix(skill_lists, skill_names)
Creates the ground-truth label matrix.

Example:

if there are 3 skills: ["Excel", "Tax", "Audit"]
and a job has ["Excel", "Audit"]
then its row becomes:

[1, 0, 1]

In [ ]:
def _safe_div(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    result = np.zeros_like(numerator, dtype=np.float64)
    np.divide(numerator, denominator, out=result, where=denominator != 0)
    return result


def _binary_f1(tp, fp, fn):
    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    return _safe_div(2.0 * precision * recall, precision + recall)


def _micro_metrics_from_counts(tp, fp, fn):
    precision = float(tp / (tp + fp)) if (tp + fp) else 0.0
    recall = float(tp / (tp + fn)) if (tp + fn) else 0.0
    f1 = float((2.0 * precision * recall) / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1

Metric helpers

np.divide(numerator, denominator, out=result, where=denominator != 0)
This is the key line.

It means:

- divide numerator / denominator
- store the answer in result
- but only at positions where denominator != 0

So if denominator is 0 somewhere, NumPy skips that division entirely. No warning, no inf, no nan. That entry in result stays 0.0

_binary_f1(tp, fp, fn)
Computes F1 from true positives, false positives, false negatives.

_micro_metrics_from_counts(tp, fp, fn)
Computes micro precision, recall, and F1 from global counts.

# Threshold Selection Logic

Global threshold

_find_best_global_threshold(similarity, labels)
tries one cutoff for the entire matrix.

The matrix similarity has one score for every (job, skill) pair.
The matrix labels has the true answer for every (job, skill) pair:

- 1 = this job really has this skill
- 0 = it does not

For each candidate threshold: pred = similarity >= threshold

That creates a full prediction matrix of True/False.

Then it counts:

- tp: predicted 1 and true 1
- fp: predicted 1 but true 0
- fn: predicted 0 but true 1

From those, it computes:

- micro precision
- micro recall
- micro F1

It also computes per-skill F1 and averages them into macro F1.

Then _is_better_candidate() decides which threshold is best, using this priority:

1. higher micro F1
2. if tied, higher recall
3. if tied, higher macro F1
4. if tied, lower threshold

So the code is mostly optimizing for overall micro F1.

Why try only unique similarity values?

Because predictions only change when the threshold passes one of the actual scores.

Example: if scores are [0.2, 0.5, 0.9], then threshold 0.51 gives the same predictions as 0.89. So it is enough to test unique score values.

In [ ]:
EPS = 1e-12

def _is_better_candidate(candidate, incumbent):
    c_f1, c_recall, c_macro_f1, c_threshold = candidate
    i_f1, i_recall, i_macro_f1, i_threshold = incumbent

    if c_f1 > i_f1 + EPS:
        return True
    if i_f1 > c_f1 + EPS:
        return False

    if c_recall > i_recall + EPS:
        return True
    if i_recall > c_recall + EPS:
        return False

    if c_macro_f1 > i_macro_f1 + EPS:
        return True
    if i_macro_f1 > c_macro_f1 + EPS:
        return False

    return c_threshold < i_threshold - EPS


def _find_best_global_threshold(similarity, labels):
    flat_scores = similarity.ravel()
    flat_labels = labels.ravel().astype(bool)
    candidates = np.unique(flat_scores)

    best = None
    best_threshold = None

    for threshold in candidates:
        pred = flat_scores >= threshold
        tp = int(np.logical_and(pred, flat_labels).sum())
        fp = int(np.logical_and(pred, ~flat_labels).sum())
        fn = int(np.logical_and(~pred, flat_labels).sum())
        _, micro_recall, micro_f1 = _micro_metrics_from_counts(tp, fp, fn)

        pred_2d = similarity >= threshold
        tp_cols = np.logical_and(pred_2d, labels == 1).sum(axis=0)
        fp_cols = np.logical_and(pred_2d, labels == 0).sum(axis=0)
        fn_cols = np.logical_and(~pred_2d, labels == 1).sum(axis=0)
        macro_f1 = float(_binary_f1(tp_cols, fp_cols, fn_cols).mean())

        candidate = (micro_f1, micro_recall, macro_f1, float(threshold))
        if best is None or _is_better_candidate(candidate, best):
            best = candidate
            best_threshold = float(threshold)

    return best_threshold

# Training & Evaluation Pipeline

Step 1: Load data
- reads the job CSV with uuid, title, extracted_skills
- reads job embeddings JSONL
- reads skill embeddings JSONL

Step 3: Build matrices
- job_matrix: job embeddings
- skill_matrix: skill embeddings
- similarity: cosine similarity matrix
- labels: binary true-label matrix

Step 4: Select threshold
It finds one best global threshold and applies it to all skills.

Step 5: Final evaluation

After applying the global threshold, it computes:

- micro precision / recall / F1
- macro precision / recall / F1
- and stores them in metrics_df.

Step 6: Build outputs

It returns 3 DataFrames:

`metrics_df`
overall evaluation metrics

`thresholds_df`
per-skill table showing the shared global threshold and each skill's positive support

`predictions_df`
- per-job actual vs predicted skills
- counts of TP / FP / FN

In [ ]:
def fit_acc_similarity(
    job_csv="data/acc/audit_tax_accounting_jobs.csv",
    job_embeddings_jsonl="embedding/jobs_train_embeddings.jsonl",
    skill_embeddings_jsonl="embedding/acc_skills_embeddings.jsonl",
):
    labels_df = pd.read_csv(job_csv, usecols=["uuid", "title", "extracted_skills"])

    job_records = load_jsonl(job_embeddings_jsonl)
    skill_records = load_jsonl(skill_embeddings_jsonl)

    jobs_df = pd.DataFrame(
        {
            "job_id": [record["job_id"] for record in job_records],
            "embedded_title": [record.get("title", "") for record in job_records],
            "job_embedding": [record["embedding"] for record in job_records],
        }
    )

    merged = jobs_df.merge(
        labels_df,
        left_on="job_id",
        right_on="uuid",
        how="left",
    )

    skill_names = [record["skill_name"] for record in skill_records]

    actual_skill_lists = merged["extracted_skills"].map(parse_skill_list).tolist()

    job_matrix = np.vstack(merged["job_embedding"].to_numpy()).astype(np.float32)
    skill_matrix = np.vstack([record["embedding"] for record in skill_records]).astype(np.float32)
    similarity = job_matrix @ skill_matrix.T

    labels = build_indicator_matrix(actual_skill_lists, skill_names)
    if similarity.shape != labels.shape:
        raise ValueError(
            f"Similarity matrix shape {similarity.shape} does not match label matrix shape {labels.shape}"
        )

    global_threshold = _find_best_global_threshold(similarity, labels)
    thresholds = np.full(len(skill_names), global_threshold, dtype=np.float32)
    positive_support = labels.sum(axis=0).astype(int)

    predictions = similarity >= global_threshold
    positives = labels == 1
    tp_cols = np.logical_and(predictions, positives).sum(axis=0).astype(np.int64)
    fp_cols = np.logical_and(predictions, ~positives).sum(axis=0).astype(np.int64)
    fn_cols = np.logical_and(~predictions, positives).sum(axis=0).astype(np.int64)

    total_tp = int(tp_cols.sum())
    total_fp = int(fp_cols.sum())
    total_fn = int(fn_cols.sum())

    micro_precision, micro_recall, micro_f1 = _micro_metrics_from_counts(total_tp, total_fp, total_fn)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    metrics_df = pd.DataFrame(
        [
            {
                "num_jobs": len(merged),
                "num_skills": len(skill_names),
                "global_threshold": global_threshold,
                "micro_precision": micro_precision,
                "micro_recall": micro_recall,
                "micro_f1": micro_f1,
                "macro_precision": float(macro_precision),
                "macro_recall": float(macro_recall),
                "macro_f1": float(macro_f1),
            }
        ]
    )

    thresholds_df = pd.DataFrame(
        {
            "skill_name": skill_names,
            "threshold": thresholds.astype(float),
            "positive_support": positive_support,
        }
    )

    predicted_skill_lists = [
        [skill_names[idx] for idx, is_on in enumerate(row) if is_on]
        for row in predictions
    ]
    predictions_df = pd.DataFrame(
        {
            "job_id": merged["job_id"],
            "title": merged["title"].fillna(merged["embedded_title"]).astype(str),
            "actual_skills": [" | ".join(skills) for skills in actual_skill_lists],
            "predicted_skills": [" | ".join(skills) for skills in predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(actual_skill_lists, predicted_skill_lists)],
        }
    )

    print(f"Evaluated {len(merged)} jobs against {len(skill_names)} ACC skills.")
    print(metrics_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    print(f"\nGlobal threshold applied to all skills: {global_threshold:.4f}")

    return metrics_df, thresholds_df, predictions_df


# Final Excecution Block

runs the whole pipeline.

In [ ]:
metrics_df, thresholds_df, predictions_df = fit_acc_similarity(
    job_csv="data/acc/audit_tax_accounting_jobs.csv",
    job_embeddings_jsonl="embedding/jobs_train_embeddings.jsonl",
    skill_embeddings_jsonl="embedding/acc_skills_embeddings.jsonl",
)

# Optional inspection
print("\nPredictions preview:")
print(predictions_df.head(5).to_string(index=False))